In [147]:
from importlib import import_module

named_libs = [('pandas', 'pd'), ('datetime', 'dt')] # (library_name, shorthand)
for (name, short) in named_libs:
    try:
        lib = import_module(name)
    except:
        print(sys.exc_info())
    else:
        globals()[short] = lib  

libnames = ['requests', 'json', 'pyexasol', 'configparser']
for libname in libnames:
    try:
        lib = import_module(libname)
    except:
        print(sys.exc_info())
    else:
        globals()[libname] = lib
        
### API EXTRACTION

def cleanNsafeExtract():
    try:
       ### Initialize the dataframe
        final_df = pd.DataFrame()
        harmonized_df = pd.DataFrame()
        miss_dict = {              
          '1':'Public health authorities guidelines'
        , '2':'Risk assessment'
        , '3':'Risk assessment - hotstaff'
        , '4':'Action plan'
        , '5':'Log books'
        , '6':'Staff/Guests well informed'
        , '7':'Staff/Guests assessed'
        , '8':'Pictograms'
        , '9':'Updated contact info'
        , '10':'Staff training'
        , '11':'Tracing system'
        , '12':'Hygiene - suppliers/contractors'
        , '13':'Web checkin'
        , '14':'Hothygiene - guests'
        , '15':'Common area- social distancing'
        , '16':'PPE/kits - staff'
        , '17':'PPE/kits - guests'
        , '18':'Staff remind guests'
        , '19':'Disinfectant - room keys'
        , '20':'Disinfected pools'
        , '21':'Lockers - Spa/fitness social distancing'
        , '22':'Lockers - Spa/fitness'
        , '23':'Disinfectant products - fitness'
        , '24':'Disinfectant reminder - fitness'
        , '25':'Fitness equipment'
        , '26':'Social distancing - Spa/fitness'
        , '27':'Dish washers and washing machine'
        , '28':'HVAC systems'
        , '29':'Sanitiser dispensers'
        , '30':'Public restrooms'
        , '31':'Take away/Room service'
        , '32':'Safety measures - buffets'
        , '33':'Disinfection - buffet areas'
        , '34':'Disinfection - vending machines'
        , '35':'Dishwashing machine'
        , '36':'Social distancing dining'
        , '37':'Social distancing seating'
        , '38':'Guest reminder'
        , '39':'Cleaning protocol - public areas'
        , '40':'Disinfection protocol - COVID cases'
        , '41':'Disinfection - guest rooms'
        , '42':'Dirty linen storage'
        , '43':'Ventilation'
        , '44':'Staff to access meeting room'
        , '45':'Luggage store for group'
        , '46':'Social distancing - meeting rooms'
        , '':''
        }
        ### API link
        url = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=500"
        parsed = json.loads(requests.get(url).text)
        ### Get the number of pages through -> #int(parsed['total_pages'])
        for x in range(parsed['total_pages']):
            url_iter = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page="+str(x+1)+"&size=500"
            response = requests.get(url_iter)
            if response.status_code == 200 or response.status_code == 403:    
                data = response.text
                parsed = json.loads(data)
                df = pd.DataFrame(parsed.get('results'))
                final_df = final_df.append(df, ignore_index=True)
            else: 
                print("Request failed page: {} ".format(x))
        final_df['created_date']= final_df['created_date'].astype(str).str[:-6]
        final_df['updated_date']= final_df['updated_date'].astype(str).str[:-6]
        final_df['audit_date']= final_df['audit_date'].astype(str).str[:-6]
        final_df['created_date']= pd.to_datetime(final_df['created_date'], utc=False)
        final_df['updated_date']= pd.to_datetime(final_df['updated_date'], utc=False)
        final_df['audit_date']= pd.to_datetime(final_df['audit_date'], utc=False)
        harmonized_df = final_df[final_df.hkey.notnull()]
        harmonized_df =harmonized_df.reset_index()

        # Removing duplicates in missed column
        for index, row in harmonized_df.iterrows():
            if row.missed != '':
                string = ",".join(str(x).strip() for x in list(set(row['missed'].split(','))))
                harmonized_df.loc[index, 'missed'] = string
            else:
                harmonized_df.loc[index, 'missed'] = ''

        # Creating a description column for the missed values
        missed_desc = []
        for row in harmonized_df.itertuples(name='missed'): 
            temp = row.missed.split(',')
            missed_desc.append([miss_dict[temp[i]] for i in range(len(temp))])
        temp = [str(item) for item in missed_desc]
        missed_df = pd.DataFrame(temp)
        harmonized_df = harmonized_df.join(missed_df)
        harmonized_df.rename(columns={0: "missed_desc"}, inplace = True)
        harmonized_df['missed_desc'][harmonized_df.missed_desc.str.len() < 5] = None
        harmonized_df.drop(['index'], axis= 1, inplace = True)
    except Exception as e:
        print('Failed in function cleanNsafe - ')
        raise e
    return harmonized_df, print(f'Number of records having HOTEL_IDs {final_df.id[final_df.hkey.notnull()].count()}.'), print(f'Number of records with no HOTEL_IDs  {final_df.id[final_df.hkey.isna()].count()}.')

In [143]:
def cleanNsafeLoad(df):
    harmonized_df = pd.DataFrame()
    harmonized_df = df
    try:
        #Location of the ini file
        config = configparser.ConfigParser()
        ## Config location CHANGE
        config.read('C:\\Users\\USER\\.spyder-py3\\pfxPROD.ini')
        dsn=config['pfxPROD']['dsn']
        user=config['pfxPROD']['user']
        pwd=config['pfxPROD']['pwd']
        schema=config['pfxPROD']['schema']
        # Exasol connection
        connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
        connect.execute("TRUNCATE TABLE DWHPFX.CLEAN_SAFE_HOTELS")
        connect.import_from_pandas(harmonized_df, table = ('DWHPFX','CLEAN_SAFE_HOTELS'))
        stmt = connect.last_statement()
        print(f'Number of records inserted  {stmt.rowcount()}.')
    except:
        print('Failed in function cleanNsafeLoad - check DB connection')
#     return print(f'Number of records inserted  {harmonized_df.id[harmonized_df.hkey.notnull()].count()}.')



In [148]:
df, a, b = cleanNsafeExtract()
cleanNsafeLoad(df)
df.head()

In [79]:
from urllib.parse import urlsplit
parsed = urlsplit("https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100")
print('query  :', parsed.query)

In [ ]:
from datetime import date
df.to_excel('C:\\Users\\USER\\Documents\\misc\\'+str(date.today())+'_test.xlsx', 
              sheet_name='Sheet1', 
              header=True,
              encoding='utf-8',
              index=False,
              freeze_panes=(1,0) )